# Rebuild Corpus with Full Trial Text

**Why.** The existing `ctmatch_ir` corpus (`doc_texts.txt` + dense embeddings) is built from
**eligibility-criteria text only** (confirmed: README line 21, and the eligibility-rule text in
the indexed docs). Patient queries are rich diagnostic narratives; eligibility rules often don't
even name the disease. This mismatch is a major, fixable cause of the low full-corpus retrieval
recall (BM25 recall@1000 = 0.1144 on TREC22).

This notebook re-fetches the **full descriptive fields** for the exact 374,647 corpus NCT IDs
from the ClinicalTrials.gov API v2 and rebuilds the document text as:

> `brief_title` + `official_title` + `conditions` + `brief_summary` + `detailed_description`
> + `interventions` + `eligibility_criteria`

The title / condition / summary fields are what actually match a patient's presenting narrative.

**No original 2021-04-27 archive needed** — we already have the canonical NCT-ID list
(`index2docid.txt`) and fetch by ID. NCT IDs are stable; descriptive fields (title, condition,
summary) are also stable over time, so live-API vintage drift is negligible for retrieval.

**Outputs (Drive):**
- `doc_fulltext.jsonl` — one record per NCT ID (all parsed fields), resumable checkpoint
- `doc_texts_fulltext.txt` — one doc string per line, **aligned to `index2docid.txt` order**
  (drop-in replacement for `doc_texts.txt`)

**Next:** point `eval_fullcorpus.ipynb` at the new corpus, and re-encode dense embeddings from
the full text (`reembed_corpus.ipynb`) — both retrievers were handicapped by the old text.

**Runtime:** ~20-40 min for 374k IDs at batch=100 (CPU/network only, no GPU).

In [ ]:
!pip install -q requests datasets tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_ROOT     = '/content/drive/MyDrive/ct_data23'
FULLTEXT_JSONL = f'{DATA_ROOT}/doc_fulltext.jsonl'        # per-trial parsed fields (checkpoint)
DOC_TEXTS_OUT  = f'{DATA_ROOT}/doc_texts_fulltext.txt'    # aligned to index2docid order

API_BASE      = 'https://clinicaltrials.gov/api/v2/studies'
BATCH_SIZE    = 100     # NCT IDs per filter.ids request (lower to 50 if requests fail)
REQ_PAUSE     = 0.10    # seconds between requests (polite)
MAX_RETRIES   = 4

# Field paths to request (keeps payload small). nctId is REQUIRED so parse_study can key
# each record; without it every record parses to an empty id. If the API rejects these,
# blank FIELDS to fetch full records — the sanity cell will make a failure obvious.
FIELD_PATHS = ','.join([
    'protocolSection.identificationModule.nctId',
    'protocolSection.identificationModule.briefTitle',
    'protocolSection.identificationModule.officialTitle',
    'protocolSection.conditionsModule.conditions',
    'protocolSection.descriptionModule.briefSummary',
    'protocolSection.descriptionModule.detailedDescription',
    'protocolSection.armsInterventionsModule.interventions',
    'protocolSection.eligibilityModule.eligibilityCriteria',
])
print('config set')

In [ ]:
from datasets import load_dataset

# Canonical corpus NCT IDs, in index2docid order (so the rebuilt corpus stays aligned to
# the existing index / embeddings pipeline).
idx_ds = load_dataset('semaj83/ctmatch_ir', data_files='index2docid.txt', split='train')
corpus_ids = [r['text'].strip() for r in idx_ds]
print(f'Corpus NCT IDs: {len(corpus_ids):,}')
print('sample:', corpus_ids[:3])

In [ ]:
import requests, time, json

session = requests.Session()

def fetch_batch(nct_ids):
    """Fetch a batch of studies by NCT ID via filter.ids. Returns list of study JSON dicts."""
    params = {
        'filter.ids': ','.join(nct_ids),
        'pageSize':   len(nct_ids),
        'format':     'json',
    }
    if FIELD_PATHS:
        params['fields'] = FIELD_PATHS
    for attempt in range(MAX_RETRIES):
        try:
            resp = session.get(API_BASE, params=params, timeout=30)
            resp.raise_for_status()
            return resp.json().get('studies', [])
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                print(f'  batch failed ({nct_ids[0]}..): {e}')
                return None
            time.sleep(2 ** attempt)
    return None

def parse_study(study):
    """Extract descriptive fields from a study JSON into a flat record."""
    ps   = study.get('protocolSection', {})
    idm  = ps.get('identificationModule', {})
    cond = ps.get('conditionsModule', {})
    desc = ps.get('descriptionModule', {})
    arms = ps.get('armsInterventionsModule', {})
    elig = ps.get('eligibilityModule', {})
    return {
        'nct_id':        idm.get('nctId', ''),
        'brief_title':   idm.get('briefTitle', ''),
        'official_title': idm.get('officialTitle', ''),
        'conditions':    cond.get('conditions', []),
        'brief_summary': desc.get('briefSummary', ''),
        'detailed_desc': desc.get('detailedDescription', ''),
        'interventions': [f"{i.get('type','')} {i.get('name','')}".strip()
                          for i in arms.get('interventions', [])],
        'eligibility':   elig.get('eligibilityCriteria', ''),
    }

def build_doc_text(rec):
    """Assemble the retrieval document string from parsed fields."""
    parts = []
    if rec['brief_title']:    parts.append(rec['brief_title'])
    if rec['official_title'] and rec['official_title'] != rec['brief_title']:
        parts.append(rec['official_title'])
    if rec['conditions']:     parts.append('Conditions: ' + '; '.join(rec['conditions']))
    if rec['brief_summary']:  parts.append(rec['brief_summary'])
    if rec['detailed_desc']:  parts.append(rec['detailed_desc'])
    if rec['interventions']:  parts.append('Interventions: ' + '; '.join(rec['interventions']))
    if rec['eligibility']:    parts.append('Eligibility: ' + rec['eligibility'])
    return '\n'.join(parts)

print('fetch/parse functions ready')

## Sanity check — one batch

Confirm the API returns all requested IDs and that the assembled doc text now contains
title/condition/summary (not just eligibility). **If fields come back empty, clear `FIELDS`
in config and re-run** — the API may want full records rather than field paths.

In [ ]:
_test = corpus_ids[:BATCH_SIZE]
_studies = fetch_batch(_test)
print(f'Requested {len(_test)} IDs, API returned {len(_studies) if _studies else 0} studies\n')

if _studies:
    rec = parse_study(_studies[0])
    print(f"=== {rec['nct_id']} ===")
    print('brief_title  :', rec['brief_title'][:100])
    print('conditions   :', rec['conditions'])
    print('has summary  :', bool(rec['brief_summary']))
    print('has elig     :', bool(rec['eligibility']))
    print('\n--- assembled doc text (first 700 chars) ---')
    print(build_doc_text(rec)[:700])
    returned = {parse_study(s)['nct_id'] for s in _studies}
    missing = [i for i in _test if i not in returned]
    print(f'\nIDs requested but not returned: {len(missing)} {missing[:5]}')

## Full fetch (resumable)
Appends parsed records to `doc_fulltext.jsonl`; re-running skips already-fetched IDs.

In [ ]:
import os
from tqdm.auto import tqdm

done = set()
if os.path.exists(FULLTEXT_JSONL):
    with open(FULLTEXT_JSONL) as f:
        for line in f:
            try:
                done.add(json.loads(line)['nct_id'])
            except Exception:
                pass
todo = [i for i in corpus_ids if i not in done]
print(f'Done: {len(done):,}  |  Remaining: {len(todo):,}')

n_ok, n_missing = 0, 0
with open(FULLTEXT_JSONL, 'a') as out_f:
    for start in tqdm(range(0, len(todo), BATCH_SIZE), desc='Fetching'):
        batch = todo[start:start+BATCH_SIZE]
        studies = fetch_batch(batch)
        if studies is None:
            continue  # whole batch failed — retried on next run
        returned = set()
        for s in studies:
            rec = parse_study(s)
            if not rec['nct_id']:
                continue
            out_f.write(json.dumps(rec) + '\n')
            returned.add(rec['nct_id'])
            n_ok += 1
        # record IDs the API had no data for, so we don't re-fetch them forever
        for i in batch:
            if i not in returned:
                out_f.write(json.dumps({'nct_id': i, 'missing': True}) + '\n')
                n_missing += 1
        time.sleep(REQ_PAUSE)

print(f'\nFetched this run — ok: {n_ok:,}  |  missing from API: {n_missing:,}')

## Assemble aligned corpus
Emit one doc string per line in **`index2docid.txt` order**. Missing/empty trials get an
empty line (kept as placeholders so row *i* still maps to `corpus_ids[i]`).

In [ ]:
# Load all parsed records into a dict
id2text = {}
n_missing = 0
with open(FULLTEXT_JSONL) as f:
    for line in f:
        rec = json.loads(line)
        if rec.get('missing'):
            id2text[rec['nct_id']] = ''
            n_missing += 1
        else:
            id2text[rec['nct_id']] = build_doc_text(rec)

# Emit in corpus order; newlines within docs are replaced so one doc == one line
n_empty = 0
with open(DOC_TEXTS_OUT, 'w') as out_f:
    for nct_id in corpus_ids:
        text = id2text.get(nct_id, '').replace('\n', ' ').replace('\r', ' ').strip()
        if not text:
            n_empty += 1
        out_f.write(text + '\n')

lengths = [len(id2text.get(i, '')) for i in corpus_ids]
import numpy as np
print(f'Wrote {len(corpus_ids):,} lines -> {DOC_TEXTS_OUT}')
print(f'Empty docs (missing/no fields): {n_empty:,} ({100*n_empty/len(corpus_ids):.1f}%)')
print(f'Doc length chars — median: {int(np.median(lengths))}, mean: {int(np.mean(lengths))}')
print('\nCompare to old eligibility-only corpus (~950 chars/doc). Full-text should be notably longer.')

In [ ]:
# Spot-check a doc that we know a TREC22 query should match (Kallmann-syndrome-style topic 1)
print('Example rebuilt docs:')
for nct_id in corpus_ids[:2]:
    print(f'\n=== {nct_id} ===')
    print(id2text.get(nct_id, '(missing)')[:500])